# Exercise 5.3 — Frame Potential and the Clifford 3-Design

**Chapter 5: Unitary Designs** &nbsp;|&nbsp; *Exercises*

---


## Background

The Clifford group sits at the centre of this book's ledger of moments. It is an exact
3-design, so it reproduces every Haar average of degree $(3,3)$ or lower: mean purities,
collision probabilities, and -- crucially -- the *variance* of classical-shadow estimators,
which is a third-moment quantity. Yet it is efficiently classically simulable
(Gottesman-Knill), and it is blind to magic.

Both facts have a single cause, and this notebook computes it: the Clifford group's
commutant agrees with the Haar commutant up to $t=3$ and departs from it at $t=4$.


## Motivation

Why should a *group* be a design at all, and why should the property stop at $t=3$?

The whole question reduces to counting, via one identity:

$$F^{(t)}_{G} \;=\; \dim\bigl(\text{commutant of }\{\hat W^{\otimes t}\}_{\hat W \in G}\bigr).$$

The frame potential **is** the commutant dimension. A subgroup can only have a *larger*
commutant than the full unitary group, so $F^{(t)}_G \ge F^{(t)}_{\mathrm{Haar}}$ always,
with equality exactly when $G$ is a $t$-design. Every number below is a dimension.


## Preliminaries / Toolbox

**1. Group invariance.** For a finite group $G$, the map $\hat V \mapsto \hat U^\dagger \hat V$
permutes $G$, so the double average collapses:
$$F^{(t)}_{G} = \frac{1}{|G|}\sum_{\hat W \in G} |\Tr \hat W|^{2t}.$$
This step uses closure and fails for a non-group ensemble.

**2. The commutant identity.** Since $\Tr(\hat W^{\otimes t}) = (\Tr\hat W)^t$,
$$|\Tr\hat W|^{2t} = \Tr\bigl(\hat W^{\otimes t}\otimes \hat W^{*\otimes t}\bigr),$$
so the group average is a projector and its trace is its rank.

**3. Haar reference.** By Schur-Weyl the Haar commutant is spanned by the $t!$ permutation
operators, independent iff $D \ge t$. So
$$F^{(t)}_{\mathrm{Haar}} = \sum_{\lambda \vdash t,\; \ell(\lambda) \le D} (f^{\lambda})^2,$$
which equals $t!$ when $D \ge t$, and drops below it otherwise.


## Part 1 -- The Haar reference

We need $F^{(t)}_{\mathrm{Haar}}$ before we can say what "agreeing with Haar" means.
Note the two special cases: $t!$ once $D \ge t$, and the **Catalan numbers** at $D=2$ --
the same sequence that counted non-crossing pairings in Chapter 2, because $\ell(\lambda)\le2$
is exactly a non-crossing condition.


In [ ]:
import numpy as np
from math import factorial

def f_lambda(lam, t):
    """Dimension of the S_t irrep for partition lam, by the hook-length formula."""
    cells = [(i, j) for i, r in enumerate(lam) for j in range(r)]
    prod = 1
    for (i, j) in cells:
        arm = lam[i] - j - 1
        leg = sum(1 for k in range(i + 1, len(lam)) if lam[k] > j)
        prod *= (arm + leg + 1)
    return factorial(t) // prod

def partitions(n, maxpart=None):
    if maxpart is None: maxpart = n
    if n == 0: yield []
    for k in range(min(n, maxpart), 0, -1):
        for rest in partitions(n - k, k):
            yield [k] + rest

def F_haar(t, D):
    """Commutant dimension of U(D)^{otimes t} = sum over partitions with <= D rows."""
    return sum(f_lambda(p, t) ** 2 for p in partitions(t) if len(p) <= D)

def catalan(n): return factorial(2*n) // (factorial(n) * factorial(n + 1))

print(f"{'t':>3} {'D=2':>5} {'D=3':>5} {'D=4':>5} {'D=8':>5} | {'t!':>5} {'C_t':>5}")
for t in range(1, 6):
    row = [F_haar(t, D) for D in (2, 3, 4, 8)]
    print(f"{t:>3} {row[0]:>5} {row[1]:>5} {row[2]:>5} {row[3]:>5} | {factorial(t):>5} {catalan(t):>5}")

# the two special cases, asserted
for t in range(1, 6):
    assert F_haar(t, D=t)   == factorial(t), "D >= t must give t!"
    assert F_haar(t, D=2)   == catalan(t),   "D = 2 must give the Catalan number"
print("\nOK: F_haar = t! for D >= t, and = C_t for D = 2.")


**Caution.** It is tempting to say "$t!$ for $D\ge t$, Catalan for $D<t$". That is **false**.
At $t=4$, $D=3$ the answer is $23$, not $C_4=14$: the only partition excluded is $(1,1,1,1)$,
so we lose $1$ from $24$. Catalan is a $D=2$ coincidence, not a general $D<t$ rule.


In [ ]:
assert F_haar(4, 3) == 23, "D=3, t=4 is 23"
assert catalan(4)  == 14
print("F_haar(t=4, D=3) =", F_haar(4, 3), " (not the Catalan number 14)")
print("excluded partition at D=3: (1,1,1,1) with f^lambda = 1  ->  24 - 1 = 23")


## Part 2 -- One qubit, by hand

Modulo phase, every single-qubit Clifford is a Bloch-sphere rotation preserving the
octahedron, so $|\Tr\hat W|^2 = 4\cos^2(\theta/2)$. Classifying the 24 elements by
rotation angle gives the histogram, and part 1 of the exercise then gives $F^{(t)}$
in closed form:

$$F^{(t)} = \frac{1}{24}\left[1\cdot 4^{t} + 6\cdot 2^{t} + 8\cdot 1^{t} + 9\cdot 0\right].$$


In [ ]:
I2 = np.eye(2, dtype=complex)
X  = np.array([[0, 1], [1, 0]], dtype=complex)
Z  = np.array([[1, 0], [0, -1]], dtype=complex)
H  = (X + Z) / np.sqrt(2)
S  = np.array([[1, 0], [0, 1j]], dtype=complex)

def canon(U):
    """Fix the global phase so group elements can be compared."""
    f = U.flatten()
    i = int(np.argmax(np.abs(f) > 1e-8))
    return np.round(U * np.exp(-1j * np.angle(f[i])), 6) + 0.0

def generate(gens, dim):
    """Breadth-first closure of <gens>, modulo global phase."""
    start = canon(np.eye(dim, dtype=complex))
    G = {start.tobytes(): start}
    frontier = [start]
    while frontier:
        nxt = []
        for U in frontier:
            for g in gens:
                V = canon(g @ U)
                if V.tobytes() not in G:
                    G[V.tobytes()] = V
                    nxt.append(V)
        frontier = nxt
    return list(G.values())

C1 = generate([H, S], 2)
print("|Cl(2)| mod phase =", len(C1))
assert len(C1) == 24

from collections import Counter
hist = Counter(int(round(abs(np.trace(W))**2)) for W in C1)
print("\n|Tr W|^2 histogram:", dict(sorted(hist.items(), reverse=True)))
assert hist == {4: 1, 2: 6, 1: 8, 0: 9}, "octahedral classes"

# closed form from the histogram, vs direct average
for t in range(1, 5):
    closed = (1*4**t + 6*2**t + 8*1**t) / 24
    direct = np.mean([abs(np.trace(W))**(2*t) for W in C1])
    # canon() rounds to 6 dp, so compare with a relative tolerance
    assert np.isclose(closed, direct, rtol=1e-5), (closed, direct)
    print(f"  t={t}:  F^(t) = {closed:>6.3f}   (Haar D=2: {F_haar(t,2)})")


## Part 3 -- Two qubits, by enumeration

$|\mathrm{Cl}(4)| = 11{,}520$, small enough to enumerate exactly. Now $D=4\ge t$ for all
$t\le4$, so the Haar reference is the clean $t!$ and the comparison is sharper than at $N=1$.


In [ ]:
def kron(*ms):
    out = np.array([[1]], dtype=complex)
    for m in ms: out = np.kron(out, m)
    return out

CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
gens2 = [kron(H, I2), kron(I2, H), kron(S, I2), kron(I2, S), CNOT]

C2 = generate(gens2, 4)
print("|Cl(4)| mod phase =", len(C2))
assert len(C2) == 11520

tr2 = np.array([abs(np.trace(W))**2 for W in C2])
print("|Tr W|^2 values:", sorted(set(np.round(tr2, 6))))

print(f"\n{'t':>3} {'F_Cliff':>9} {'F_Haar=t!':>10} {'excess':>7}")
for t in range(1, 5):
    F = (tr2**t).mean()
    print(f"{t:>3} {F:>9.3f} {F_haar(t,4):>10} {F - F_haar(t,4):>7.3f}")

for t in (1, 2, 3):
    assert np.isclose((tr2**t).mean(), F_haar(t, 4), rtol=1e-5), "3-design: must match Haar"
assert np.isclose((tr2**4).mean(), 29, rtol=1e-5), "t=4 gives 29"
print("\nOK: matches Haar for t <= 3; F^(4) = 29 > 24.")


## Part 4 -- The ladder in $N$, and why it saturates

Direct enumeration stops at $N=2$: $|\mathrm{Cl}(8)| = 92{,}897{,}280$.

For general $N$ the Clifford commutant is spanned not by permutations but by operators
labelled by *stochastic Lagrangian subspaces*, and their number is

$$\prod_{k=0}^{t-2}\left(2^{k}+1\right) \;=\; 1,\,2,\,6,\,30,\,270,\ldots$$

against $t! = 1,2,6,24,120$ for Haar. But a spanning set has that dimension only if its
members are **independent**, which holds for $N \ge t-1$. That is the whole explanation of
the small-$N$ ladder -- and it is the same degeneracy that puts poles in the Weingarten
function when $D < k$.


In [ ]:
def clifford_commutant_count(t):
    """Number of spanning operators for the Clifford commutant (large-N value)."""
    p = 1
    for k in range(0, t - 1): p *= (2**k + 1)
    return p

print(f"{'t':>3} {'Clifford span':>14} {'Haar t!':>9}")
for t in range(1, 6):
    print(f"{t:>3} {clifford_commutant_count(t):>14} {factorial(t):>9}")
for t in (1, 2, 3):
    assert clifford_commutant_count(t) == factorial(t), "must agree for t <= 3"
assert clifford_commutant_count(4) == 30 != factorial(4)

print("\nThe measured ladder at t = 4:")
print(f"  N=1  (D=2):  F(4) = 15   Haar = {F_haar(4,2):>2}   excess = {15-F_haar(4,2)}   <- Haar side also degenerate (D<t)")
print(f"  N=2  (D=4):  F(4) = 29   Haar = {F_haar(4,4):>2}   excess = {29-F_haar(4,4)}   <- one relation survives (N < t-1)")
print(f"  N>=3 (D>=8): F(4) = 30   Haar = {F_haar(4,8):>2}   excess = {30-F_haar(4,8)}   <- saturated, constant thereafter")
print("\nThreshold at t=4 is N >= t-1 = 3.")


## Conclusion

The Clifford group is an **exact 3-design** and **not a 4-design**, uniformly in $N$.

Because the frame potential *is* a commutant dimension, the excess at $t=4$ is a **count**:
it is the number of invariants a Clifford twirl preserves and a Haar twirl destroys. Those
invariants first appear at the fourth moment, so no first-, second- or third-moment quantity
can see them -- any such diagnostic is by construction blind to the Clifford/Haar difference.

The **stabilizer Rényi entropy** is the cheapest instrument that is *not* blind, which is
exactly why magic is a fourth-moment quantity and why it is invisible to Clifford circuits.
This is the Gottesman-Knill theorem seen from the moment hierarchy.

One hypothesis worth keeping: this is a **qubit** statement. For qudits of odd prime-power
dimension the Clifford group is still an exact 2-design but already fails at $t=3$. The
3-design property is a coincidence of $d=2$, and the practicality of Clifford-based shadow
tomography rests on it.
